# BacteriaID-CNN Pipeline Walkthrough

Runs one sample Gram-stained image through every stage of the CV pipeline
(RGB->HSV, thresholding, Canny, morphology, connected components, region
extraction) before it ever reaches the CNN. Useful as a sanity check after
downloading the dataset, and as a source of report figures.

Run `preprocessing/segmentation.py validate --raw-dir dataset/raw` first to
confirm the dataset is laid out correctly.

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))
import config
from preprocessing import rgb_to_hsv, thresholding, edge_detection, morphology
from preprocessing.segmentation import extract_regions

## 1. Load one sample raw image

In [ ]:
species_dirs = sorted(d for d in config.RAW_DIR.iterdir() if d.is_dir())
assert species_dirs, f"No class folders found in {config.RAW_DIR} -- extract the Mendeley dataset there first."

sample_species = species_dirs[0]
sample_path = sorted(sample_species.glob('*'))[0]
print(f"Sample: {sample_path}")

bgr = cv2.imread(str(sample_path))
plt.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
plt.title(f"Original: {sample_species.name}")
plt.axis('off')

## 2. RGB -> HSV conversion

In [ ]:
hsv = rgb_to_hsv.bgr_to_hsv(bgr)
h, s, v = rgb_to_hsv.split_hsv_channels(hsv)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, channel, title in zip(axes, [h, s, v], ['Hue', 'Saturation', 'Value']):
    ax.imshow(channel, cmap='gray')
    ax.set_title(title)
    ax.axis('off')

## 3. Image thresholding (Otsu + adaptive)

In [ ]:
gray = rgb_to_hsv.get_segmentation_channel(hsv, 'saturation')
_, otsu_mask = thresholding.otsu_threshold(gray)
adaptive_mask = thresholding.adaptive_threshold(gray)
combined_thresh = thresholding.combine_thresholds(otsu_mask, adaptive_mask, mode='or')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, mask, title in zip(axes, [otsu_mask, adaptive_mask, combined_thresh], ['Otsu', 'Adaptive', 'Combined']):
    ax.imshow(mask, cmap='gray')
    ax.set_title(title)
    ax.axis('off')

## 4. Canny edge detection

In [ ]:
edges = edge_detection.auto_canny(gray)
edge_mask = edge_detection.edges_to_mask(edges)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(edges, cmap='gray'); axes[0].set_title('Canny edges'); axes[0].axis('off')
axes[1].imshow(edge_mask, cmap='gray'); axes[1].set_title('Dilated edge mask'); axes[1].axis('off')

## 5. Morphological cleanup

In [ ]:
combined_mask = cv2.bitwise_or(combined_thresh, edge_mask)
cleaned_mask = morphology.clean_mask(combined_mask)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(combined_mask, cmap='gray'); axes[0].set_title('Threshold + edges (pre-morphology)'); axes[0].axis('off')
axes[1].imshow(cleaned_mask, cmap='gray'); axes[1].set_title('After close+open'); axes[1].axis('off')

## 6. Connected components + region extraction

In [ ]:
regions = extract_regions(bgr, cleaned_mask, output_size=config.IMAGE_SIZE, strategy='largest')
print(f"Found {len(regions)} region(s)")

bbox_image = bgr.copy()
for crop, stats in regions:
    x, y, w, h = stats['x'], stats['y'], stats['w'], stats['h']
    cv2.rectangle(bbox_image, (x, y), (x + w, y + h), (0, 255, 0), 3)

fig, axes = plt.subplots(1, 1 + len(regions), figsize=(5 * (1 + len(regions)), 5))
if len(regions) == 0:
    axes = [axes]
axes[0].imshow(cv2.cvtColor(bbox_image, cv2.COLOR_BGR2RGB)); axes[0].set_title('Detected region(s)'); axes[0].axis('off')
for ax, (crop, stats) in zip(axes[1:], regions):
    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Crop (area={stats['area']})")
    ax.axis('off')

## 7. Next steps

Once this looks reasonable across a few sample species, run the full pipeline:

```
python -m preprocessing.segmentation all
python -m models.train --data-dir dataset/raw --run-name raw_v1
python -m models.train --data-dir dataset/segmented --run-name segmented_v1
```

## 8. Optional: run a trained model on this sample (if one exists yet)

In [ ]:
import json
import numpy as np

try:
    from tensorflow import keras

    run_dir = config.RESULTS_DIR / 'segmented_v1'
    model = keras.models.load_model(run_dir / 'final_model.keras')
    with open(run_dir / 'class_names.json') as f:
        class_names = json.load(f)

    crop, _stats = regions[0]
    batch = np.expand_dims(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB), axis=0)
    prediction = model.predict(batch)[0]
    predicted_idx = int(np.argmax(prediction))
    print(f"Predicted: {class_names[predicted_idx]} ({prediction[predicted_idx]:.2%} confidence)")
except FileNotFoundError:
    print("No trained model found yet at results/segmented_v1/final_model.keras -- run models/train.py first.")